# Scenario Dreamer Dev / Report Eval (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/scenario-dreamer-decision-layer/blob/main/notebooks/scenario_dreamer_dev_eval_colab.ipynb)

Use this notebook after the smoke baseline is already working.

Goals:
- lock the stock dev-tier baseline,
- persist artifacts directly to Drive,
- optionally run the report tier once the stack is stable.


In [ ]:
# Step 1: Sync this repo into the Colab runtime
import os
import subprocess
import sys

REPO_URL = 'https://github.com/achyutmorang/scenario-dreamer-decision-layer.git'
REPO_DIR = '/content/scenario-dreamer-decision-layer'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

print('repo_rev:', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# Suggested defaults for Drive-backed dev / report evaluation
%env SCENARIO_DREAMER_DRIVE_ROOT=/content/drive/MyDrive/scenario_dreamer_research
%env SCENARIO_DREAMER_RUN_SETUP=1
%env SCENARIO_DREAMER_RUN_DEV=1
%env SCENARIO_DREAMER_RUN_REPORT=0
%env SCENARIO_DREAMER_WRITE_TRANSFER_HOOK=0


In [ ]:
# Step 2: Mount Drive, clone/pin upstream Scenario Dreamer, and bind Drive-backed assets/results
import json
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive', force_remount=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR], check=True)

from scenario_dreamer_decision_layer.colab import bind_drive_layout, inspect_bound_layout

subprocess.run([sys.executable, 'scripts/bootstrap_baseline.py', '--clone-upstream', '--write-lock'], check=True)
binding = bind_drive_layout(os.environ['SCENARIO_DREAMER_DRIVE_ROOT'])
print('binding:')
print(json.dumps(binding, indent=2, sort_keys=True))
print('inspection:')
print(json.dumps(inspect_bound_layout(), indent=2, sort_keys=True))


In [ ]:
# Step 3: Install a lean runtime for Scenario Dreamer simulation on Colab
import os
import subprocess
import sys

RUN_SETUP = os.environ.get('SCENARIO_DREAMER_RUN_SETUP', '1').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_SETUP:', RUN_SETUP)
if RUN_SETUP:
    subprocess.run([sys.executable, 'scripts/setup_colab_runtime.py', '--editable-project'], check=True)
else:
    print('Skipping runtime setup.')


In [ ]:
# Step 4: Dry-run the dev tier (and optionally the report tier) before execution
import json
import os
import subprocess
import sys

raw = subprocess.check_output([sys.executable, 'scripts/run_dev_eval.py', '--dry-run'], text=True)
print(raw)
dev_dry_run = json.loads(raw)
dev_missing_assets = dev_dry_run.get('missing_assets', {})
dev_scenario_count = int(dev_dry_run.get('scenario_count', 0))
print('dev_dry_run_summary:')
print(json.dumps({
    'scenario_count': dev_scenario_count,
    'missing_assets': dev_missing_assets,
    'config_snapshot': dev_dry_run.get('config_snapshot', ''),
}, indent=2, sort_keys=True))
if dev_missing_assets:
    raise RuntimeError(f'Resolve missing assets before Step 5: {dev_missing_assets}')
if dev_scenario_count <= 0:
    raise RuntimeError('Dev dry-run found zero scenarios. Verify the Scenario Dreamer environment pack binding before Step 5.')

RUN_REPORT = os.environ.get('SCENARIO_DREAMER_RUN_REPORT', '0').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_REPORT:', RUN_REPORT)
if RUN_REPORT:
    raw_report = subprocess.check_output([sys.executable, 'scripts/run_report_eval.py', '--dry-run'], text=True)
    print(raw_report)
    report_dry_run = json.loads(raw_report)
    print('report_dry_run_summary:')
    print(json.dumps({
        'scenario_count': int(report_dry_run.get('scenario_count', 0)),
        'missing_assets': report_dry_run.get('missing_assets', {}),
        'config_snapshot': report_dry_run.get('config_snapshot', ''),
    }, indent=2, sort_keys=True))


In [ ]:
# Step 5: Execute the dev tier and optionally the report tier
import json
import os
import subprocess
import sys


def _run_json_cmd(cmd, label):
    proc = subprocess.run(cmd, text=True, capture_output=True, check=False)
    stdout = proc.stdout.strip()
    stderr = proc.stderr.strip()
    if stdout:
        print(stdout)
    if stderr:
        print(f'[{label}_stderr]')
        print(stderr)
    if proc.returncode != 0:
        diag = {
            'exit_code': proc.returncode,
            'failed_command': ' '.join(cmd),
            'results_root': os.environ.get('SCENARIO_DREAMER_RESULTS_ROOT', ''),
            'recent_stdout': stdout[-4000:],
            'recent_stderr': stderr[-4000:],
        }
        print(f'{label}_diagnostics:')
        print(json.dumps(diag, indent=2, sort_keys=True))
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    try:
        return json.loads(stdout)
    except json.JSONDecodeError as exc:
        diag = {
            'failed_command': ' '.join(cmd),
            'results_root': os.environ.get('SCENARIO_DREAMER_RESULTS_ROOT', ''),
            'recent_stdout': stdout[-4000:],
            'recent_stderr': stderr[-4000:],
            'parse_error': str(exc),
        }
        print(f'{label}_diagnostics:')
        print(json.dumps(diag, indent=2, sort_keys=True))
        raise


RUN_DEV = os.environ.get('SCENARIO_DREAMER_RUN_DEV', '1').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
RUN_REPORT = os.environ.get('SCENARIO_DREAMER_RUN_REPORT', '0').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
WRITE_TRANSFER_HOOK = os.environ.get('SCENARIO_DREAMER_WRITE_TRANSFER_HOOK', '0').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_DEV:', RUN_DEV)
print('RUN_REPORT:', RUN_REPORT)
print('WRITE_TRANSFER_HOOK:', WRITE_TRANSFER_HOOK)

dev_payload = None
report_payload = None
if RUN_DEV:
    dev_payload = _run_json_cmd([sys.executable, 'scripts/run_dev_eval.py'], 'step5_dev')
    print('dev_summary:')
    print(json.dumps({
        'run_id': dev_payload.get('run_id', ''),
        'run_dir': dev_payload.get('run_dir', ''),
        'scenario_count': dev_payload.get('scenario_count', 0),
        'metrics': dev_payload.get('metrics', {}),
    }, indent=2, sort_keys=True))
else:
    print('Skipping dev run.')

if RUN_REPORT:
    cmd = [sys.executable, 'scripts/run_report_eval.py']
    if WRITE_TRANSFER_HOOK:
        cmd.append('--write-transfer-hook')
    report_payload = _run_json_cmd(cmd, 'step5_report')
    print('report_summary:')
    print(json.dumps({
        'run_id': report_payload.get('run_id', ''),
        'run_dir': report_payload.get('run_dir', ''),
        'scenario_count': report_payload.get('scenario_count', 0),
        'metrics': report_payload.get('metrics', {}),
        'transfer_hook_request': report_payload.get('transfer_hook_request', ''),
    }, indent=2, sort_keys=True))
else:
    print('Skipping report run.')


In [ ]:
# Step 6: Inspect the latest Drive-backed artifacts
import json
import os
from pathlib import Path

results_root = Path(os.environ['SCENARIO_DREAMER_RESULTS_ROOT'])
run_dirs = sorted([p for p in results_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
print('results_root:', results_root)
print('num_run_dirs:', len(run_dirs))
for run_dir in run_dirs[:3]:
    print('run_dir:', run_dir)
    for name in ['run_manifest.json', 'metrics.json', 'config_snapshot.json', 'transfer_hook_request.json']:
        path = run_dir / name
        if path.exists():
            print(f'{path.name}:')
            print(path.read_text()[:4000])
